# Meme Reaction V2 — Master Plan de Upgrades

**Fecha de inicio:** 2026-05-30  
**Ubicación:** `automatizaciones/Meme_Reaction_V2/`  
**Referencia V1:** `automatizaciones/Meme_Reaction/RESUMEN_Y_MEJORAS_V2`

---

## Filosofía V2

- Pipeline que pueda correr **lo más desatendido posible** para los casos claros
- Lo que no sea claro va a **colas de revisión manual** (no bloquea el pipeline)
- **SQLite** como base de datos central (reemplaza todos los JSONs)
- **Branching inteligente**: fotos directas → IA automática | screenshots de video → revisión manual
- **GPT-4o** se queda como modelo principal (no modelo local). Inversión: ~$9/mes a escala
- **Selenium se queda** (funciona bien), pero se mejora para multi-perfil y solo posts nuevos
- **Instagram only** por ahora (multi-plataforma queda como futuro)

---

## Resumen Rápido de Decisiones

| Mejora | Estado | Notas |
| --- | --- | --- |
| A1-A5 (Optimizaciones) | ✅ TODAS | Config JSON, logging, retry, paralelismo, cache |
| B1 (Upload social) | ✅ SÍ | + auto-generación de título/descripción por IA |
| B2 (Scheduler) | ⏸️ FUTURO | Necesita servidor. Telegram notifs sí cuando esté listo |
| B3 (Analytics) | 💡 IDEA | No prioritario ahora |
| B4 (Clip finder) | 🔄 MODIFICADO | Semi-manual: guarda pendientes + ejemplos para YouTube |
| B5 (Batch review) | ✅ SÍ | Grid visual HTML/Tkinter |
| C1 (Eliminar Selenium) | ❌ NO | Se queda. Mejoras: multi-perfil + solo posts nuevos |
| C2 (Event-driven) | ✅ SÍ | Con colas + branching + script de status |
| C3 (Auto paso 4) | ✅ SÍ | >90% auto-accept, <40% skip, medio = cola |
| C4 (Multi-plataforma) | ⏸️ FUTURO | Solo IG por ahora |
| C5 (SQLite) | ✅ SÍ | Reemplaza JSONs. Investigar conexión con GitHub |
| C6 (Modelo local) | ❌ NO | GPT-4o se queda. Mejorar prompts es la prioridad |
| D1 (Templates video) | ⏸️ DESPUÉS | Primero pulir template actual |
| D2 (Transiciones) | ❌ NO | No |
| D3 (Audio inteligente) | ⏸️ FUTURO | Feature futura |
| D4 (A/B Testing) | ✅ MODIFICADO | Generar variantes, usuario escoge cuál subir |
| D5 (Caption timing) | ⏸️ FUTURO | No funciona bien aún, pulir primero |
| E1 (Rate limit budget) | ✅ SÍ | Con SQLite |
| E2 (Health checks) | ✅ SÍ | Rápido de implementar |
| E3 (Versionado prompts) | ✅ MODIFICADO | + historial de feedback del usuario para mejorar prompts |
| E4 (Backup) | ✅ SÍ | Con SQLite + GitHub |

## Cambios Generales (aplican a TODOS los scripts)

Estos cambios son transversales. Se implementan una vez y afectan todo el pipeline.

---

### 1. SQLite como base de datos central
**Reemplaza:** Todos los JSONs en `historial/`  
**Archivo:** `meme_reaction.db` (en raíz de Meme_Reaction_V2)  
**Tablas propuestas:**
- `memes` (shortcode, source_profile, source_type, likes, status, timestamps...)
- `clasificaciones` (shortcode FK, categorias, descripcion, confianza, ideas_video...)
- `clips` (id, descripcion, categorias, filename, usado_count...)
- `matches` (shortcode FK, clip_id FK, accuracy, caption, status...)
- `videos_generados` (shortcode FK, output_path, config_json, uploaded...)
- `uploads` (video FK, platform, url, timestamp, engagement...)
- `prompt_versions` (id, prompt_text, fecha, accuracy_promedio...)
- `user_feedback` (id, shortcode, step, feedback_text, timestamp...)
- `rate_limits` (api, requests_today, tokens_today, last_reset...)
- `pipeline_runs` (run_id, paso, status, duration, error_msg...)

**💡 PENDIENTE POR RESOLVER:** ¿Cómo conectar SQLite con GitHub?
- Opción A: El .db se commitea al repo (funciona si es pequeño <50MB)
- Opción B: Solo se commitean las migraciones SQL (schema), y cada máquina tiene su .db local
- Opción C: Exportar a CSVs periódicamente y esos sí van a git
- **Recomendación**: Opción A para empezar (el .db será pequeño). Si crece, migrar a B.

---

### 2. Config centralizado (`config.json`)
**Archivo:** `config.json` en raíz de Meme_Reaction_V2  
**Contenido:**
```json
{
  "perfiles_target": ["elmello2023", "otro_perfil"],
  "scraping": {
    "scroll_count": 8,
    "scroll_delay": 3.0,
    "delay_entre_perfiles": 5
  },
  "descarga": {
    "min_likes": 5000,
    "max_por_sesion": 50,
    "delay_entre_posts": 5
  },
  "clasificacion": {
    "modelo": "gpt-4o",
    "max_tokens": 1800,
    "max_por_sesion": 20,
    "paralelismo": 3,
    "categorias": ["humor_absurdo", "humor_dark", "cringe", "..."]
  },
  "match": {
    "modelo": "gpt-4o-mini",
    "auto_accept_threshold": 90,
    "auto_skip_threshold": 40,
    "max_por_sesion": 10
  },
  "video": {
    "width": 1080,
    "height": 1920,
    "fps": 30,
    "meme_max_ratio": 0.70,
    "clip_max_ratio": 0.40
  },
  "telegram": {
    "bot_token": "ENV:TELEGRAM_BOT_TOKEN",
    "chat_id": "ENV:TELEGRAM_CHAT_ID",
    "notificar_video_generado": true,
    "notificar_upload": true
  },
  "dry_run": false
}
```

---

### 3. Logging estructurado
**Módulo:** `utils/logger.py`  
- Reemplaza todos los `print()` por `logging.info/warning/error`
- Log file rotativo: `logs/run_{timestamp}.log`
- Métricas automáticas: tiempo por paso, tokens gastados, posts procesados
- Nivel configurable en config.json

---

### 4. Retry automático con backoff
**Módulo:** `utils/retry.py`  
- Decorator `@with_retry(max_attempts=3, backoff_factor=2)`
- Para requests a IG: retry en timeout/5xx
- Para OpenAI: retry si JSON inválido (baja temperatura en retry)
- main.py: opción `--retry-failed` que solo re-ejecuta pasos que fallaron

---

### 5. Health checks al inicio
**Módulo:** `utils/health.py`  
- Verifica: .env tiene OPENAI_API_KEY, Brave existe, ffmpeg accesible
- Verifica: SQLite db existe y schema es correcto
- Si algo falla: mensaje claro + exit(1) antes de desperdiciar tiempo
- Auto-cleanup de temporales

---

### 6. Rate limit budget
**Tabla SQLite:** `rate_limits`  
- Trackea requests/tokens a OpenAI y requests a IG por día
- Antes de cada request: verifica si queda budget
- Si se acerca al límite: warning. Si se excede: pausa automática
- Reset diario automático

## Scraping V2 — Dos Modos de Operación

**Scripts:** `1a_scrape_inicial.py` + `1b_scrape_nuevos.py`  
**Tecnología:** Selenium + Brave (se mantiene)

---

### Modo A: Scrape Inicial (`1a_scrape_inicial.py`)
**Cuándo se usa:** Primera vez que agregas un perfil nuevo  
**Flujo:**
1. Usuario pone el @username como argumento
2. Selenium abre Brave, pausa para login manual
3. Navega al perfil y scrollea MUCHO (configurable, default 20+ scrolls)
4. Captura TODOS los shortcodes del grid
5. Los registra en SQLite como `status='por_descargar'`

**Uso:** `python 1a_scrape_inicial.py --perfil elmello2023 --scrolls 30`

---

### Modo B: Scrape de Nuevos Posts (`1b_scrape_nuevos.py`)
**Cuándo se usa:** Ejecución regular/periódica  
**Flujo:**
1. Lee `perfiles_target` de config.json (puede ser 100+ perfiles)
2. Selenium abre Brave, pausa para login manual
3. Para cada perfil:
   - Navega al perfil
   - Scrollea POCO (2-3 scrolls, solo los posts recientes)
   - Captura shortcodes
   - Compara con SQLite: ¿cuáles son NUEVOS?
   - Solo guarda los nuevos como `status='por_descargar'`
   - Si encuentra 0 nuevos, pasa al siguiente perfil rápido
4. Resumen al final: X perfiles visitados, Y posts nuevos encontrados

**Uso:** `python 1b_scrape_nuevos.py`  
**Optimización:** Si un perfil no tiene posts nuevos en 3 ejecuciones seguidas, reducir frecuencia de visita.

---

### Diferencias vs V1
| Aspecto | V1 | V2 |
| --- | --- | --- |
| Perfiles | Hardcoded en script | config.json (dinámico) |
| Modo | Uno solo (scrapea todo) | Dos modos: inicial + nuevos |
| Detección nuevos | Compara con JSON historial | Compara con SQLite |
| Multi-perfil | Sí pero rudimentario | Optimizado (skip si no hay nuevos) |
| Escala | ~5 perfiles | Diseñado para 100+ |

## Pipeline Branching — Fotos vs Screenshots de Video

**Concepto nuevo en V2:** El pipeline se BIFURCA después de la descarga.

---

### Flujo con Branching

```
Paso 1 (scrape) → Paso 2 (descarga)
                        │
                        ├── GraphImage (foto directa)
                        │       │
                        │       └──→ Paso 3 (Clasificación IA) → auto
                        │
                        ├── GraphVideo (screenshot de frame)
                        │       │
                        │       └──→ Cola de REVISIÓN MANUAL
                        │              (batch_review.py)
                        │              Si aprobado → Paso 3
                        │
                        └── GraphSidecar (carousel)
                                │
                                └──→ SKIP (igual que V1)
```

### Razón
- Los screenshots de video son impredecibles: muchos son frames inútiles que gastan tokens de OpenAI
- Las fotos directas casi siempre son memes reales → pueden ir directo a IA
- El paso 2 ya sabe el tipo (GraphImage vs GraphVideo) → marca en SQLite: `source_type='foto'` o `source_type='frame'`

### En SQLite
- Campo `source_type` en tabla `memes`: 'foto' | 'frame' | 'carousel'
- Campo `status`: 'por_descargar' → 'descargado' → 'pendiente_review' | 'listo_clasificar' → 'clasificado' → ...
- Fotos: status pasa directo de 'descargado' a 'listo_clasificar'
- Frames: status pasa a 'pendiente_review' hasta que batch_review lo apruebe

## Script de Status — Resumen del Pipeline

**Archivo:** `status.py`  
**Propósito:** Vista rápida de dónde está todo. Un solo comando para saber qué hay pendiente.

---

### Output ejemplo
```
============================================================
   MEME REACTION V2 - STATUS
============================================================
   📥 Por descargar:         47 posts
   🖼️  Descargados (foto):     23
   🎥 Descargados (frame):    18
   
   👁️  Pendientes review:      18 frames
   ✅ Listos para clasificar: 23 (fotos) + 5 (frames aprobados)
   
   🧠 Clasificados:           20
   🎬 Match pendiente:        15 (esperando matching)
   🎬 Match hecho (>90%):     8 (auto-aceptados)
   🎬 Match hecho (40-90%):   4 (en cola de revisión)
   🎬 Sin clip (<40%):        3 (buscar clip manual)
   
   🎥 Videos por generar:     8
   🎥 Videos generados:       5
   📤 Por subir:              5
   📤 Subidos:                2
============================================================
```

### Implementación
- Queries simples a SQLite por status de cada tabla
- Puede correr en cualquier momento sin afectar nada
- Opción: `--detailed` para ver desglose por perfil
- Opción: integrarlo con Telegram (resumen diario automático)

---

### Cuándo se ejecuta
- Manualmente cuando quieras saber el estado
- Automáticamente al final de cada paso del pipeline
- (Futuro) Como parte del scheduler/Telegram bot

## Batch Review — Grid Visual de Memes

**Archivo:** `batch_review.py`  
**Reemplaza:** `revisar_memes.py` (abrir imagen por imagen)  
**Prioridad:** ALTA (ahorra mucho tiempo)

---

### Concepto
En vez de abrir 1 imagen a la vez y decidir sí/no en terminal, mostrar un GRID con todos los memes pendientes y aprobar/rechazar con un click.

### Opciones de implementación

**Opción A: Página HTML estática (RECOMENDADA)**
- Genera un HTML con thumbnails de todas las imágenes pendientes
- Cada imagen tiene botón ✅ / ❌
- Al final: botón "Guardar decisiones" que genera un JSON
- El script lee ese JSON y actualiza SQLite
- **Ventaja**: Sin dependencias extra, funciona en cualquier navegador
- **Ventaja**: Puede mostrar metadata (likes, source_type, perfil)

**Opción B: Tkinter (más interactivo)**
- Ventana con grid scroll-able
- Click izquierdo = aprobar, click derecho = rechazar
- Más inmediato pero requiere Tkinter

### Flujo
1. `python batch_review.py` → genera `review_page.html` + abre en navegador
2. Usuario ve grid, hace click en aprobar/rechazar cada uno
3. Click "Guardar" → escribe `review_results.json`
4. Script lee el JSON → actualiza SQLite (status de cada meme)
5. Borra el HTML temporal

### Filtra qué mostrar
- Default: solo memes con `status='pendiente_review'` (frames de video)
- Flag `--all`: muestra TODOS los descargados sin clasificar
- Flag `--reclasificar`: muestra los ya clasificados para re-evaluar

### Info por thumbnail
- Imagen (150x150px)
- Shortcode
- Likes / Comments
- Source type (foto/frame)
- Perfil de origen

## Match V2 — Automatización del Paso 4

**Archivo:** `4_match_clip.py` (reescrito)  
**Cambio principal:** Ya no es 100% interactivo. Tiene auto-accept para matches claros.

---

### Nuevos Thresholds (configurable en config.json)

| Accuracy | Acción | Resultado |
| --- | --- | --- |
| ≥ 90% | Auto-accept | Se marca como matched, pasa a generación |
| 40% - 89% | Cola de revisión | Espera confirmación humana (no bloquea) |
| < 40% | Auto-skip | Se marca como `buscar_clip` (sin clip adecuado) |

### Flujo V2
```
Meme clasificado
      │
      ├── ≥ 90% match con clip del catálogo
      │       └──→ Auto-accept → cola de generación
      │
      ├── 40-89% match
      │       └──→ Cola de revisión humana
      │           (próxima ejecución interactiva)
      │
      └── < 40% (ningún clip sirve)
              └──→ Cola de "buscar clip"
                  (clip_finder_manual.py)
```

### Modo interactivo (para cola de revisión)
- Igual que V1: la IA sugiere, tú decides
- Pero SOLO para los que están en cola (no todos)
- Los auto-accepted ya pasaron sin intervención

---

## Clip Finder Manual (`clip_finder_manual.py`)

**Propósito:** Ayudarte a encontrar clips para memes que no tienen match  
**Flujo:**
1. Lee de SQLite todos los memes con `status='buscar_clip'`
2. Para cada uno muestra:
   - La imagen del meme
   - La descripción/categorías
   - Lo que la IA sugirió como clip ideal (del paso 3: `ideas_video.clip_ideal`)
   - 3-5 ejemplos concretos de búsqueda en YouTube (generados por IA)
3. Tú buscas en YouTube, descargas el clip (manual o con yt-dlp)
4. Lo catalogas con `catalogar_clips.py`
5. Próxima vez que corre el match: ese meme ya tiene un clip disponible

**Ejemplo de output:**
```
────────────────────────────────────────
[1/3] Shortcode: ABC123
Categorías: humor_dark, plot_twist
Clip ideal según IA: "Persona que se ríe nerviosamente y luego se queda seria"

Búsquedas sugeridas para YouTube:
  1. "nervous laugh meme compilation"
  2. "hold up wait a minute reaction"
  3. "guy laughing then stops meme original"
  4. "Michael Jordan laughing then serious face"

[Enter para siguiente / 'd' cuando lo tengas]
────────────────────────────────────────
```

## Upload Social (`9_upload_social.py`)

**Prioridad:** Alta (es el objetivo final del pipeline)  
**Plataformas:** TikTok, Instagram Reels, YouTube Shorts

---

### Funcionalidad Core
1. Lee de SQLite todos los videos con `status='por_subir'`
2. Para cada video, la IA genera automáticamente:
   - **Título** (optimizado para cada plataforma)
   - **Descripción** (con hashtags relevantes)
   - **Tags/categoría** de la plataforma
3. Muestra preview: video + metadata propuesta
4. Usuario confirma o edita
5. Sube a las plataformas seleccionadas
6. Registra URL + timestamp en SQLite
7. (Si Telegram activado) Envía notificación: "🎥 Video subido: [link]"

### APIs a usar
- **YouTube Shorts**: YouTube Data API v3 (OAuth 2.0)
- **TikTok**: TikTok Content Posting API
- **Instagram Reels**: instagrapi (ya es dependencia) o Meta Graph API

### Generación de metadata con IA
- Input: imagen del meme + categorías + caption del video
- Output: título, descripción, hashtags (por plataforma)
- Modelo: GPT-4o-mini (texto, barato)
- **DETALLE A DEFINIR DESPUÉS**: Prompt exacto, estilo de títulos, hashtags favoritos, etc.

### Variantes de video (D4 modificado)
- Cuando se generan 2-3 variantes del mismo meme (diferente clip/caption)
- El script muestra las variantes lado a lado
- Usuario escoge cuál se sube
- Las demás se archivan (no se borran, por si acaso)

---

### 💡 PENDIENTE: Scheduler + Servidor
**Problema:** Todo esto requiere máquina encendida.  
**Ideas a explorar (NO ahora, cuando el pipeline esté completo):**
- Rentar VPS barato (DigitalOcean $5/mes, Oracle Cloud free tier)
- GitHub Actions con schedule (para pasos que no necesitan Selenium)
- Railway / Render para scripts headless
- Raspberry Pi en casa (siempre encendida)

**Telegram notificaciones (SÍ implementar desde ya):**
- Bot de Telegram que avisa cuando:
  - Se generan videos nuevos
  - Se suben videos
  - Hay errores en el pipeline
  - Resumen diario de status
- Librería: `python-telegram-bot` o requests directos a la API

## Versionado de Prompts + Feedback Loop

**Tablas SQLite:** `prompt_versions` + `user_feedback`  
**Concepto:** Mejora continua de prompts basada en tu feedback real.

---

### Cómo funciona

#### 1. Registro de versiones de prompt
Cada vez que se cambia un prompt (clasificación, match, caption):
- Se guarda la versión anterior + la nueva en SQLite
- Se marca con fecha y un tag descriptivo ("v3: agregué categoría sus")
- Cada clasificación/match registra QUÉ versión de prompt usó

#### 2. Captura de feedback del usuario
Durante el paso 4 (match interactivo) ya existe la opción 'r' (responder).  
**V2 mejora:**
- TODO lo que escribes en modo 'r' se guarda en SQLite:
  - Tu sugerencia: "yo creo que queda mejor X porque Y"
  - La evaluación de la IA: "accuracy 85%"
  - Tu decisión final
- También durante review de clasificación (paso 3.5):
  - Si marcas MAL + nota: se guarda el shortcode, lo que dijo la IA, y tu corrección
- Durante batch_review:
  - Si rechazas algo que la IA hubiera clasificado como válido: feedback implícito

#### 3. Exportar feedback para mejorar prompts
**Script:** `export_feedback.py`  
- Genera un resumen legible de todo el feedback acumulado:
  - "De 50 clasificaciones, 8 fueron marcadas MAL. Patrón: la IA confunde X con Y"
  - "En matching, el usuario prefiere clips de [tipo] sobre [tipo]"
  - "Frases exactas del usuario: ['no, mejor este porque...', 'este clip no queda...']"
- Este resumen se comparte conmigo (o con GPT) para generar un prompt mejorado
- Se prueba la nueva versión y se compara accuracy vs la anterior

#### 4. Ciclo completo
```
Prompt v1 → Clasificación/Match → Review + Feedback → Export → Prompt v2 → ...
```

---

### Ejemplo de tabla `user_feedback`
| id | shortcode | step | ia_said | user_said | decision | timestamp |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | ABC123 | match | "clip_laugh, 75%" | "no, mejor el de Michael Jordan" | override:MJ_clip | 2026-05-30 |
| 2 | DEF456 | classify | "humor_dark, 0.9" | "esto es cringe, no dark" | corrected:cringe | 2026-05-30 |
| 3 | GHI789 | review | "valido: true" | (rechazado en batch) | rejected | 2026-05-30 |

## Event-Driven Pipeline — Colas basadas en SQLite

**Concepto:** Cada paso lee de SQLite qué hay pendiente y procesa solo eso.  
**Ya no es:** "ejecutar 1→2→3→..." obligatorio en orden.  
**Ahora es:** "ejecuta lo que puedas, lo demás espera en cola."

---

### Status Machine por meme

```
por_descargar
    │
    └→ descargado
          │
          ├→ [foto] → listo_clasificar
          │
          └→ [frame] → pendiente_review
                          │
                          ├→ aprobado → listo_clasificar
                          └→ rechazado (fin)

listo_clasificar
    │
    └→ clasificado
          │
          ├→ [valido=false] → descartado_ia (fin)
          └→ [valido=true] → pendiente_match

pendiente_match
    │
    ├→ [≥90%] → matched_auto → por_generar
    ├→ [40-89%] → match_review (cola humana)
    └→ [<40%] → buscar_clip (cola para clip_finder)

match_review
    │
    ├→ confirmado → por_generar
    └→ rechazado → buscar_clip

por_generar
    │
    └→ generado → por_subir → subido (fin)
```

### Cómo se ejecuta en la práctica

Cada script mira su cola:
- `2_download.py`: busca `status='por_descargar'`
- `batch_review.py`: busca `status='pendiente_review'`
- `3_classify.py`: busca `status='listo_clasificar'`
- `4_match.py`: busca `status='pendiente_match'`
- `7_generate.py`: busca `status='por_generar'`
- `9_upload.py`: busca `status='por_subir'`

Puedes ejecutar cualquiera en cualquier orden. Si no tiene pendientes, sale rápido.

---

### main.py V2
Sigue existiendo como orquestador secuencial:  
`python main.py` = ejecuta todo lo que se pueda en orden  
Pero cada paso es independiente y puede correrse solo.

## Paralelismo en Clasificación (Paso 3)

**Cambio:** De secuencial (1 imagen a la vez) a paralelo (3-5 simultáneas)  
**Tecnología:** `asyncio` + `openai` async client  
**Impacto:** Reduce tiempo de ~2s/meme a ~0.5s/meme efectivo

---

### Implementación
```python
# Pseudocódigo
import asyncio
from openai import AsyncOpenAI

async def classify_batch(memes, concurrency=3):
    semaphore = asyncio.Semaphore(concurrency)
    client = AsyncOpenAI()
    
    async def classify_one(meme):
        async with semaphore:
            # ... llamada a GPT-4o Vision ...
            pass
    
    tasks = [classify_one(m) for m in memes]
    results = await asyncio.gather(*tasks)
    return results
```

### Rate limits a respetar
- OpenAI Tier 1: 500 RPM, 30,000 TPM
- Con `concurrency=3` y 2s por request: ~90 RPM (well within limits)
- Configurable en config.json: `clasificacion.paralelismo`

### Cache inteligente
- Antes de clasificar: calcula hash SHA-256 del archivo de imagen
- Si ese hash ya existe en SQLite con clasificación → skip (no gastar tokens)
- Útil para: re-runs, imágenes duplicadas de diferentes perfiles

## Estructura de Carpetas V2

```
automatizaciones/Meme_Reaction_V2/
├── new version upgrades        # Este notebook (documentación)
├── config.json                  # Config centralizado
├── meme_reaction.db             # SQLite (reemplaza todos los JSONs)
├── requirements.txt             # Deps del proyecto
├── main.py                      # Orquestador (ejecuta todo en orden)
├── status.py                    # Resumen del estado del pipeline
│
├── utils/                       # Módulos compartidos
│   ├── __init__.py
│   ├── db.py                    # Conexión + helpers SQLite
│   ├── config.py                # Carga config.json
│   ├── logger.py                # Logging estructurado
│   ├── retry.py                 # Decorator de retry con backoff
│   ├── health.py                # Health checks
│   ├── rate_limiter.py          # Budget de rate limits
│   └── telegram.py              # Notificaciones Telegram
│
├── 1a_scrape_inicial.py         # Scrape todos los posts de 1 perfil
├── 1b_scrape_nuevos.py          # Scrape solo posts nuevos (multi-perfil)
├── 2_download_memes.py          # Descarga + branching foto/frame
├── 3_classify_meme.py           # Clasificación IA (paralela)
├── 4_match_clip.py              # Match auto + semi-auto + cola
├── 7_generate_video.py          # Generación de video
├── 9_upload_social.py           # Upload a redes + IA metadata
│
├── batch_review.py              # Grid visual HTML para review
├── clip_finder_manual.py        # Sugerencias de clips para buscar
├── catalogar_clips.py           # Catalogar clips (igual que V1)
├── export_feedback.py           # Exportar feedback para mejorar prompts
│
├── memes_descargados/           # Imágenes descargadas
├── clips/                       # Clips catalogados
├── output/                      # Videos generados
├── logs/                        # Logs por sesión
└── templates/                   # (futuro) Templates de video
```

### Scripts eliminados vs V1
- `5_verify_match.py` → NO se implementa (integrado en paso 4 con thresholds)
- `6_generate_caption.py` → NO se implementa (caption se decide en paso 4)
- `8_save_config.py` → Integrado en paso 7 (igual que V1)
- `revisar_memes.py` → Reemplazado por `batch_review.py`
- `3.5_review_clasificacion.py` → Integrado en feedback loop + batch_review
- `auto_meme_reaction.py` → Eliminado (legacy)

## Pendientes y Dudas por Resolver

Cosas que quedaron abiertas o que necesitan más detalle antes de implementar.

---

### 🟡 Por definir cuando lleguemos ahí

1. **Upload social - Prompt de título/descripción**
   - ¿Qué estilo de títulos quieres? (clickbait, descriptivo, misterioso)
   - ¿Hashtags fijos + dinámicos?
   - ¿Diferente tono por plataforma?
   - Se definirá cuando empecemos con `9_upload_social.py`

2. **SQLite + GitHub**
   - ¿Commitear el .db directo? (simple, funciona si <50MB)
   - ¿O solo migraciones SQL?
   - Recomendación: empezar commiteando el .db, migrar si crece mucho

3. **Scheduler / Servidor**
   - NO se implementa ahora
   - Primero: completar pipeline end-to-end en máquina local
   - Después: explorar VPS / Oracle Cloud / GitHub Actions
   - Telegram notificaciones SÍ van desde ya (no dependen de servidor)

4. **Variantes de video (D4)**
   - ¿Cuántas variantes generar? (2? 3?)
   - ¿Qué varía entre ellas? (diferente clip, diferente caption, diferente template?)
   - Se definirá cuando lleguemos a generación de video

---

### 🟢 Decisiones ya tomadas (no re-discutir)

- Selenium se queda (funciona bien)
- GPT-4o se queda (no modelo local)
- Solo Instagram por ahora
- No transiciones/efectos en video (simple es mejor por ahora)
- No caption timing (feature futura)
- Template actual se pule primero, templates nuevos después

---

### 🔴 Orden de implementación sugerido

**Fase 1: Infraestructura**
1. SQLite schema + `utils/db.py`
2. `config.json` + `utils/config.py`
3. `utils/logger.py` + `utils/retry.py` + `utils/health.py`
4. `status.py`

**Fase 2: Pipeline core (reescritura)**
5. `1a_scrape_inicial.py` + `1b_scrape_nuevos.py`
6. `2_download_memes.py` (con branching)
7. `batch_review.py` (HTML grid)
8. `3_classify_meme.py` (paralelo + cache)
9. `4_match_clip.py` (auto + semi-auto)
10. `7_generate_video.py`

**Fase 3: Extras**
11. `clip_finder_manual.py`
12. `9_upload_social.py`
13. `utils/telegram.py`
14. `export_feedback.py`

**Fase 4: Futuro**
- Scheduler + servidor
- Multi-plataforma scraping
- Audio inteligente
- Caption timing
- Templates de video
- Analytics dashboard

## Progress Log

---

### 30 Mayo 2026 - Dia 1: Infraestructura + Pipeline hasta Clasificacion

**Resumen:** Se construyo toda la infraestructura (Fase 1), los scripts de scraping+download (Fase 2 parcial), y se avanzo hasta clasificacion IA + QA dashboard.

---

#### Fase 1: Infraestructura (COMPLETADA)

Archivos creados:
| Archivo | Funcion |
| --- | --- |
| `utils/__init__.py` | Exports centralizados |
| `utils/db.py` | SQLite: 10 tablas, 7 indices, helpers CRUD, transacciones |
| `utils/config.py` | Carga config.json, resuelve ENV:VAR automaticamente |
| `utils/logger.py` | Logging con colores terminal, archivo rotativo, metricas sesion |
| `utils/retry.py` | Decorator generico + presets @retry_openai y @retry_instagram |
| `utils/health.py` | 8 checks pre-ejecucion (config, env, db, Brave, ffmpeg, dirs...) |
| `utils/rate_limiter.py` | Clase RateLimiter con budget diario por API |
| `utils/telegram.py` | Notificaciones (video generado, upload, errores, resumen) |
| `config.json` | Config centralizado completo (scraping, IA, video, Telegram, limits) |
| `status.py` | Dashboard del pipeline con --detailed y --telegram |
| `requirements.txt` | Dependencias del proyecto |

---

#### Fase 2: Pipeline Core (EN PROGRESO)

| Script | Status | Funcion |
| --- | --- | --- |
| `1a_scrape_inicial.py` | FUNCIONAL | Scrapeo 372 posts de @elmello2023 (20 scrolls) |
| `1b_scrape_nuevos.py` | CREADO | Listo para multi-perfil, no testeado aun |
| `2_download_memes.py` | FUNCIONAL | Descargo 60+ posts, branching foto/frame/carousel |
| `2b_preprocess.py` | FUNCIONAL | Auto-crop inteligente + borde por tipo (margin safety) |
| `batch_review.py` | FUNCIONAL | Grid HTML con SI/RE/NO (redescargar integrado) |
| `3_classify_meme.py` | CREADO | GPT-4o Vision, taxonomy 55 tags, cache por hash |
| `view_clasificados.py` | CREADO | QA dashboard HTML, filtros, feedback, reclasificar |
| `4_match_clip.py` | PENDIENTE | Siguiente despues de clasificar |

---

#### Tests ejecutados hoy

1. `python status.py` - DB inicializada, todo en 0
2. `python 1a_scrape_inicial.py --perfil elmello2023` - 372 posts scrapeados
3. `python 2_download_memes.py --max 10` - Primer batch, branching funcional
4. `python 2_download_memes.py --max 50` + mas - Total ~190 procesados
5. `python 2b_preprocess.py` - Crop + borde (fix: margin para no cortar texto)
6. `python batch_review.py` - Multiples sesiones, 6 aprobados, 187 rechazados
7. `python batch_review.py --apply` - Con opcion RE (redescargar) integrada

---

#### Bugs encontrados y resueltos

1. **UnicodeEncodeError en batch_review** - Emojis surrogate pairs. Fix: HTML entities.
2. **Botones JS no funcionan** - onclick inline + base64 gigante. Fix: funciones globales + HTTP server.
3. **Imagenes no cargan** - file:// bloqueaba. Fix: SimpleHTTPServer puerto 8765.
4. **Preprocess corta texto** - Auto-crop sin margin. Fix: retroceso configurable (--margin, default 6px).

---

#### Decisiones de diseno tomadas hoy

1. **Taxonomy expandida**: 55 tags en 6 grupos (formato, humor, narrativa, emocion, tematica, tono)
2. **Preprocess antes de review**: download -> preprocess -> batch_review -> classify
3. **3 acciones en batch_review**: SI (aprobar), RE (redescargar fresco), NO (rechazar)
4. **Fotos auto-aprobadas**: no pasan por batch_review, se revisan en view_clasificados
5. **5 ideas de video** (en vez de 3) en clasificacion
6. **Cache por hash**: si imagen duplicada, no gasta tokens
7. **Prompt versioning**: cada clasificacion registra version para reclasificacion selectiva
8. **View clasificados**: QA dashboard con feedback que se exporta para mejorar prompts

---

#### Estado final del dia

```
Total memes en DB:   372
Por descargar:       179
Listos clasificar:   6
Rechazados:          187
IG requests usados:  200/200 (agotado hoy)
OpenAI tokens:       0/100000
```

---

#### Pendiente para siguiente sesion

- Correr `python 3_classify_meme.py` con los 6 memes listos
- Verificar en `python view_clasificados.py` que la IA clasifico bien
- Buscar/preparar clips de reaccion para el match
- Crear `4_match_clip.py` + `7_generate_video.py`
- Descargar mas posts (IG budget se resetea manana)

## Taxonomy V2 - Categorias Expandidas para Clasificacion

La IA asigna 2-6 tags por meme. Los mismos tags se usan para clips (matching por interseccion).

---

### FORMATO DEL MEME (como esta construido visualmente)
| Tag | Descripcion |
| --- | --- |
| `formato_texto_arriba_imagen_abajo` | Texto en fondo solido + imagen debajo |
| `formato_solo_imagen` | Imagen sola sin texto overlay |
| `formato_texto_overlay` | Texto encima de la imagen |
| `formato_dos_paneles` | Comparacion lado a lado o arriba/abajo |
| `formato_multi_panel` | 3+ paneles (expanding brain, 4-panel, etc) |
| `formato_screenshot_chat` | Conversacion de WhatsApp/Twitter/DM |
| `formato_screenshot_tweet` | Screenshot de un tweet |
| `formato_screenshot_comentario` | Comentario de YouTube/IG/Reddit |
| `formato_reaccion_con_caption` | Imagen de reaccion + caption arriba |
| `formato_edit_shitpost` | Edicion mal hecha a proposito, deep fried, distorsionado |
| `formato_lista_ranking` | Tier list, ranking, top 5, comparaciones |

---

### TIPO DE HUMOR (que tipo de risa provoca)
| Tag | Descripcion |
| --- | --- |
| `humor_absurdo` | Sin sentido total, random, sin logica |
| `humor_dark` | Muerte, tragedia, temas taboo en tono de broma |
| `humor_sexual` | Doble sentido, referencias sexuales |
| `humor_cringe` | Verguenza ajena, incomodo |
| `humor_wholesome` | Tierno, positivo, inesperadamente dulce |
| `humor_ironia` | Dice una cosa, significa lo contrario |
| `humor_sarcasmo` | Comentario mordaz disfrazado de cumplido/pregunta |
| `humor_anti_meme` | El chiste es que no hay chiste |
| `humor_meta` | Meme sobre memes, autoconsciente |
| `humor_intelectual` | Requiere conocimiento especifico (historia, ciencia, etc) |

---

### ESTRUCTURA NARRATIVA (como funciona el chiste)
| Tag | Descripcion |
| --- | --- |
| `narrativa_plot_twist` | Setup normal, remate inesperado |
| `narrativa_expectativa_vs_realidad` | Lo que esperabas vs lo que paso |
| `narrativa_pov` | "POV: cuando..." perspectiva del viewer |
| `narrativa_nadie_absolutamente_nadie` | Formato "Nadie: / Yo:" |
| `narrativa_yo_vs_mi_cerebro` | Dialogo interno, contradiccion personal |
| `narrativa_before_after` | Transformacion/comparacion temporal |
| `narrativa_escalamiento` | Situacion que se sale de control gradualmente |
| `narrativa_confesion` | Admitir algo verguenzoso/inesperado |
| `narrativa_comparacion_falsa` | Comparar dos cosas que no tienen nada que ver |
| `narrativa_literalidad` | Tomar algo figurativo de forma literal |

---

### EMOCION/REACCION (que reaccion provoca o representa)
| Tag | Descripcion |
| --- | --- |
| `reaccion_sorpresa` | WTF, no me lo esperaba |
| `reaccion_indignacion` | Enojo comico, injusticia |
| `reaccion_tristeza_comica` | Sad pero en tono de meme |
| `reaccion_panico` | Estres, ansiedad, deadline |
| `reaccion_orgullo_culposo` | Saber que esta mal pero disfrutarlo |
| `reaccion_nostalgia` | Referencias a infancia, cosas viejas |
| `reaccion_relatable` | "Literalmente yo", muy identificable |
| `reaccion_flexeo` | Presumir algo absurdo o ironico |

---

### TEMATICA/CONTEXTO (sobre que habla)
| Tag | Descripcion |
| --- | --- |
| `tema_relaciones` | Novio/novia, ex, crush, friendzone |
| `tema_familia` | Mama, papa, abuelos, hermanos |
| `tema_trabajo` | Jefe, oficina, sueldo, lunes |
| `tema_escuela` | Examen, maestro, tarea, universidad |
| `tema_gaming` | Videojuegos, rage quit, toxicos |
| `tema_internet_cultura` | Redes sociales, influencers, trending |
| `tema_dinero` | Pobreza comica, quincena, gastos |
| `tema_comida` | Antojo, gordura, cocinar mal |
| `tema_animales` | Perro/gato haciendo algo, animal random |
| `tema_mexico_latam` | Cultura latina, regionalismos, costumbres |
| `tema_musica` | Canciones, artistas, conciertos, lyrics |
| `tema_deporte` | Futbol, gym, equipos |
| `tema_politica_light` | Politica pero sin ser pesado, satirico |
| `tema_existencial` | Crisis existencial, falta de proposito, comica |

---

### INTENSIDAD/TONO (que tan fuerte es)
| Tag | Descripcion |
| --- | --- |
| `tono_suave` | Family friendly, sin ofender |
| `tono_medio` | Puede incomodar a algunos |
| `tono_fuerte` | Controversial, no para todos |
| `tono_NSFW_light` | No explicito pero sugiere cosas |

---

### NOTAS
- La IA escoge libremente (no hay lista cerrada, puede sugerir tags nuevos si nada aplica)
- Los tags de FORMATO son obligatorios (al menos 1)
- Los tags de TIPO DE HUMOR son obligatorios (al menos 1)
- Los demas son opcionales pero se esperan 2-6 tags totales
- El match con clips busca interseccion: mas tags en comun = mejor match
- Para clips: se asignan tags de EMOCION/REACCION (que reaccion provoca el clip) + TONO

## 2b Preprocess - Crop Inteligente + Borde por Tipo

**Archivo:** `2b_preprocess.py`
**Ubicacion en pipeline:** download -> **2b_preprocess** -> batch_review -> classify
**Dependencias:** Pillow, numpy

---

### Algoritmo

#### Paso 1: Auto-Crop desde las 4 esquinas
1. Toma el color de las 4 esquinas (parche 5x5 para promediar)
2. Busca consenso entre esquinas (al menos 2 deben coincidir)
3. Desde cada borde (arriba, abajo, izquierda, derecha) avanza hacia adentro
4. Se detiene cuando la fila/columna de pixeles difiere del color de referencia
5. Recorta todo lo uniforme (barras negras, bordes blancos, padding innecesario)
6. Proteccion: nunca recorta mas del 70% de la imagen (min_content_ratio=0.3)

#### Paso 2: Deteccion de Tipo
| Tipo | Deteccion | Accion |
| --- | --- | --- |
| A (cuadro definido) | Despues del crop no hay bandas uniformes en el top | No agrega borde |
| B (texto + imagen) | Banda uniforme negra/blanca en el top (>12% altura) | Agrega borde del mismo color |

**Tipo B detallado:**
- Verifica si la parte superior es un fondo solido (negro o blanco)
- Si hay una banda uniforme >12% de la altura pero <85% = texto arriba + imagen abajo
- Agrega borde (default 20px) del MISMO color que el fondo del texto
- Resultado: el meme queda con espacio uniforme alrededor, limpio para video

#### Paso 3: Guardar
- Sobreescribe el archivo original (JPEG quality 92)
- Marca `preprocessed=1` en SQLite

---

### Uso

```bash
# Procesar todas las imagenes pendientes
python 2b_preprocess.py

# Dry run (ver que haria sin tocar archivos)
python 2b_preprocess.py --dry-run

# Re-procesar todas (ignora flag preprocessed)
python 2b_preprocess.py --force

# RESET: devolver aprobados a pendiente_review para re-procesar
python 2b_preprocess.py --reset

# Ajustar tolerancia y borde
python 2b_preprocess.py --tolerance 20 --border 25
```

---

### Flag --reset

Mueve todos los memes con `status='listo_clasificar'` de vuelta a `status='pendiente_review'`
y les pone `preprocessed=0`. Asi puedes:
1. `python 2b_preprocess.py --reset` (devolver a cola)
2. `python 2b_preprocess.py` (re-procesarlos con el nuevo algoritmo)
3. `python batch_review.py` (verlos ya limpios y re-aprobar)

---

### Flujo completo actualizado

```
download (2_download_memes.py)
    |
    v
2b_preprocess (crop + borde)    <-- NUEVO
    |
    v
batch_review (aprobar/rechazar visual)
    |
    v
3_classify_meme (GPT-4o Vision)
    |
    v
view_clasificados (QA visual)   <-- NUEVO (pendiente)
    |
    v
4_match_clip
```

---

### Tipo C (pendiente)
El usuario menciono un tercer tipo de meme que aun no recuerda.
Cuando se identifique, agregar logica aqui.